# oos_poor_window_factor_diff_aggregate_v1

## Purpose

Aggregate poor OOS windows and feature-difference diagnostics across the current production replay, the strongest recent actors, and the two drawdown/rebound reward diagnostics. This notebook follows the post-train review rule: compare first, explain second, and only then choose the next optimization mechanism.

## Experiment Governance

- Baselines: `v19reg_v22_budget_usage_bull_guard_v1_current`, `balanced_rule_residual_v19_reg`, `balanced_rule_residual_v22`.
- Direct comparison: `drawdown_rebound_reward_train_v1` and `mode_conditioned_rebound_reward_train_v2`.
- Role: `diagnose`.
- Target window: strict OOS `2020-01-02` through the latest saved daily replay, using rolling 63-trading-day windows.
- Changed layer: `model_selection` / diagnostic tooling, not a new actor or reward run.
- Acceptance criteria: produce shared bad-window clusters, stable feature deltas, and a concrete next optimization path that avoids another duplicate reward-only coefficient tweak.
- Failure mode: if clusters are fragmented or feature deltas conflict, do not open a new training branch from this evidence alone.

## External Basis

- [Qlib arXiv](https://arxiv.org/abs/2009.11189): keep the workflow in an AI-quant research platform with data, learning, backtest, and deployment surfaces.
- [Qlib GitHub](https://github.com/microsoft/qlib): keep diagnostics compatible with the existing qlib harness.
- [Probability of Backtest Overfitting](https://scholarworks.wmich.edu/math_pubs/42/): use OOS and cross-window evidence before selecting another candidate.
- [Deflated Sharpe Ratio](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551): repeated trials need multiple-testing and selection-bias discipline.
- [MetaTrader](https://arxiv.org/abs/2210.01774): market-condition-specific policy integration motivates separating participation failure from drawdown-control failure.
- [Learning to Classify and Imitate Trading Agents](https://arxiv.org/abs/2110.01325): behavior cloning is a plausible next mechanism only if bad clusters show a consistent policy imitation target.

## Historical Reflection

Closest success: the current budget-usage composer line showed that actor-derived slow/fast composition can improve selected windows without blindly retraining.

Closest failure: `drawdown_rebound_reward_train_v1` and `mode_conditioned_rebound_reward_train_v2` both improved some defensive behavior but remained underparticipation-heavy and failed promotion gates.

Why this is not a duplicate: this run does not tune another reward coefficient. It creates a reusable aggregation layer and uses it to classify shared bad windows before choosing the next experiment type.

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists():
            return path
    raise RuntimeError(f'could not find repo root from {start}')

ROOT = find_repo_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CARD_DIR = ROOT / 'runtime/model_selection/card/oos_poor_window_factor_diff_aggregate_v1'
DIAG_DIR = CARD_DIR / 'diagnostics'
AGG_DIR = CARD_DIR / 'aggregate'
CARD_DIR.mkdir(parents=True, exist_ok=True)
DIAG_DIR.mkdir(parents=True, exist_ok=True)
AGG_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT)
print(CARD_DIR)

/Users/rziiiii/.openclaw/workspace
/Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/oos_poor_window_factor_diff_aggregate_v1


In [2]:
from src.soxl_quant.app.oos_diagnostics import run_oos_diagnostic

DAILY_INPUTS = {
    'current_runtime_paper': ROOT / 'runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv',
    'balanced_rule_residual_v19_reg': ROOT / 'runtime/model/card/balanced_rule_residual_v19_reg/balanced_rule_residual_v19_reg_soxl_daily.csv',
    'balanced_rule_residual_v22': ROOT / 'runtime/model/card/balanced_rule_residual_v22/balanced_rule_residual_v22_soxl_daily.csv',
    'drawdown_rebound_reward_train_v1': ROOT / 'runtime/model/card/drawdown_rebound_reward_train_v1/drawdown_rebound_reward_train_v1_soxl_daily.csv',
    'mode_conditioned_rebound_reward_train_v2': ROOT / 'runtime/model/card/mode_conditioned_rebound_reward_train_v2/mode_conditioned_rebound_reward_train_v2_soxl_daily.csv',
}
missing = {label: str(path) for label, path in DAILY_INPUTS.items() if not path.exists()}
assert not missing, missing
pd.DataFrame([{'label': label, 'path': str(path)} for label, path in DAILY_INPUTS.items()])

,label,path
0,current_runtime_paper,/Users/rziiiii/.openclaw/workspace/runtime/pro...
1,balanced_rule_residual_v19_reg,/Users/rziiiii/.openclaw/workspace/runtime/mod...
2,balanced_rule_residual_v22,/Users/rziiiii/.openclaw/workspace/runtime/mod...
3,drawdown_rebound_reward_train_v1,/Users/rziiiii/.openclaw/workspace/runtime/mod...
4,mode_conditioned_rebound_reward_train_v2,/Users/rziiiii/.openclaw/workspace/runtime/mod...


In [3]:
EXTERNAL_REFERENCES = [
    {
        'label': 'Qlib arXiv',
        'url': 'https://arxiv.org/abs/2009.11189',
        'note': 'AI quant workflow with data, learning, backtest, and deployment surfaces',
    },
    {
        'label': 'Qlib GitHub',
        'url': 'https://github.com/microsoft/qlib',
        'note': 'repo-compatible qlib research-to-production workflow',
    },
    {
        'label': 'Probability of Backtest Overfitting',
        'url': 'https://scholarworks.wmich.edu/math_pubs/42/',
        'note': 'use OOS and cross-window evidence before selection',
    },
    {
        'label': 'Deflated Sharpe Ratio',
        'url': 'https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551',
        'note': 'multiple-testing and selection-bias discipline',
    },
    {
        'label': 'MetaTrader',
        'url': 'https://arxiv.org/abs/2210.01774',
        'note': 'market-condition-specific policy integration',
    },
    {
        'label': 'Behavior Cloning Trading Agents',
        'url': 'https://arxiv.org/abs/2110.01325',
        'note': 'behavior cloning as a supervised policy-refresh mechanism',
    },
]

manifests = []
for label, daily_csv in DAILY_INPUTS.items():
    output_dir = DIAG_DIR / label
    manifest = run_oos_diagnostic(
        daily_csv=daily_csv,
        profile=label,
        output_dir=output_dir,
        oos_start='2020-01-02',
        window_size=63,
        top_windows=12,
        top_features=30,
        target_excess=30.0,
        target_max_dd=15.0,
        external_references=EXTERNAL_REFERENCES,
    )
    manifests.append(manifest)

pd.DataFrame(manifests)[['profile', 'daily_csv', 'oos_end', 'windows', 'features', 'manifest']]

[94028:MainThread](2026-05-25 11:42:23,530) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[94028:MainThread](2026-05-25 11:42:24,253) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[94028:MainThread](2026-05-25 11:42:24,254) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[94028:MainThread](2026-05-25 11:42:26,841) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[94028:MainThread](2026-05-25 11:42:26,843) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[94028:MainThread](2026-05-25 11:42:36,990) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[94028:MainThread](2026-05-25 11:42:39,564) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[94028:MainThread](2026-05-25 11:42:39,565) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[94028:MainThread](2026-05-25 11:42:39,568) INFO - qlib.timer - [log.py:127] - Time cost: 0.004s | Init data Done


,profile,daily_csv,oos_end,windows,features,manifest
0,current_runtime_paper,/Users/rziiiii/.openclaw/workspace/runtime/pro...,2026-05-21,12,30,/Users/rziiiii/.openclaw/workspace/runtime/mod...
1,balanced_rule_residual_v19_reg,/Users/rziiiii/.openclaw/workspace/runtime/mod...,2026-05-21,12,30,/Users/rziiiii/.openclaw/workspace/runtime/mod...
2,balanced_rule_residual_v22,/Users/rziiiii/.openclaw/workspace/runtime/mod...,2026-05-21,12,30,/Users/rziiiii/.openclaw/workspace/runtime/mod...
3,drawdown_rebound_reward_train_v1,/Users/rziiiii/.openclaw/workspace/runtime/mod...,2026-05-21,12,30,/Users/rziiiii/.openclaw/workspace/runtime/mod...
4,mode_conditioned_rebound_reward_train_v2,/Users/rziiiii/.openclaw/workspace/runtime/mod...,2026-05-21,12,30,/Users/rziiiii/.openclaw/workspace/runtime/mod...


In [4]:
cmd = [
    sys.executable,
    str(ROOT / 'scripts/aggregate_oos_poor_windows.py'),
]
for label in DAILY_INPUTS:
    cmd.extend(['--input-dir', str(DIAG_DIR / label), '--label', label])
cmd.extend(['--output-dir', str(AGG_DIR), '--target-excess', '30', '--target-max-dd', '15', '--top-features', '30'])
result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True, check=True)
print(result.stdout)

clusters = pd.read_csv(AGG_DIR / 'poor_window_clusters.csv')
features = pd.read_csv(AGG_DIR / 'feature_delta_clusters.csv')
clusters.head(12)

oos_aggregate profiles=5 clusters=3 features=30 output=/Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/oos_poor_window_factor_diff_aggregate_v1/aggregate



,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C002,2025-04-04,2025-07-22,5,30,"balanced_rule_residual_v19_reg,balanced_rule_r...",-160.502243,-199.393437,5.924443,16.324924,0.519568,166.426685,203.420490,underparticipation,reentry_participation_or_behavior_clone
1,C003,2026-02-04,2026-05-21,4,18,"balanced_rule_residual_v22,current_runtime_pap...",-167.300896,-198.889223,10.842927,24.118142,0.089418,178.143823,205.310696,underparticipation,reentry_participation_or_behavior_clone
2,C001,2020-03-18,2020-06-19,4,12,"balanced_rule_residual_v19_reg,balanced_rule_r...",-158.096497,-174.787005,8.683677,13.885509,0.514949,166.780175,182.311302,underparticipation,reentry_participation_or_behavior_clone


In [5]:
features.head(20)

,feature,profile_count,mean_delta_z,mean_abs_delta_z,direction_consistency,mean_spearman_to_excess,profiles,stability_score
0,vol_of_vol_21,5,2.909053,2.909053,1.0,-0.029445,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.212323
1,tracking_error_vol_21,4,3.153269,3.153269,1.0,0.008294,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.074991
2,decay_63,5,-2.831926,2.831926,1.0,0.065458,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.074129
3,realized_vol_63,5,2.528318,2.528318,1.0,-0.015127,"balanced_rule_residual_v19_reg,balanced_rule_r...",4.530138
4,tracking_error_21,4,-2.161666,2.161666,1.0,0.090702,"balanced_rule_residual_v19_reg,balanced_rule_r...",3.479067
5,realized_vol_21,5,1.623577,1.623577,1.0,-0.003108,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.909059
6,vol_coef_21,5,1.464646,1.464646,1.0,-0.003431,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.624293
7,vix_level,4,1.270719,1.270719,1.0,0.097231,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.045143
8,jump_variation_5,5,1.125866,1.125866,1.0,0.005165,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.017282
9,realized_vol_10,5,1.111914,1.111914,1.0,-0.001980,"balanced_rule_residual_v19_reg,balanced_rule_r...",1.992282


In [6]:
summary = {
    'profiles': list(DAILY_INPUTS),
    'cluster_count': int(len(clusters)),
    'feature_count': int(len(features)),
    'top_cluster': clusters.iloc[0].to_dict() if not clusters.empty else {},
    'top_features': features.head(10).to_dict(orient='records'),
}
summary_path = CARD_DIR / 'selection.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=True, indent=2), encoding='utf-8')
summary

{'profiles': ['current_runtime_paper',
  'balanced_rule_residual_v19_reg',
  'balanced_rule_residual_v22',
  'drawdown_rebound_reward_train_v1',
  'mode_conditioned_rebound_reward_train_v2'],
 'cluster_count': 3,
 'feature_count': 30,
 'top_cluster': {'cluster_id': 'C002',
  'cluster_start': '2025-04-04',
  'cluster_end': '2025-07-22',
  'profile_count': 5,
  'window_count': 30,
  'profiles': 'balanced_rule_residual_v19_reg,balanced_rule_residual_v22,current_runtime_paper,drawdown_rebound_reward_train_v1,mode_conditioned_rebound_reward_train_v2',
  'mean_excess_return': -160.50224266855687,
  'worst_excess_return': -199.39343705994264,
  'mean_max_dd': 5.92444280283593,
  'max_max_dd': 16.3249243359336,
  'mean_avg_pos': 0.5195682966257669,
  'mean_bad_score': 166.42668547139283,
  'max_bad_score': 203.42049041719488,
  'dominant_failure_type': 'underparticipation',
  'primary_optimization_path': 'reentry_participation_or_behavior_clone'},
 'top_features': [{'feature': 'vol_of_vol_21',

In [7]:
if clusters.empty:
    conclusion = 'Decision: evidence_insufficient. No shared poor-window clusters were emitted.'
else:
    top = clusters.iloc[0]
    top_feature_names = ', '.join(features.head(6)['feature'].astype(str).tolist()) if not features.empty else 'none'
    decision = 'next_training_target_reentry_participation_or_behavior_clone'
    if top['dominant_failure_type'] == 'drawdown_control':
        decision = 'next_training_target_drawdown_cap_or_deleverage'
    elif top['dominant_failure_type'] == 'selection_bias':
        decision = 'next_training_target_checkpoint_or_model_selection'
    conclusion_lines = [
        '## Runtime Conclusion',
        '',
        f'Decision: `{decision}`.',
        '',
        'Top shared bad-window cluster: '
        f"`{top['cluster_start']}` .. `{top['cluster_end']}`, "
        f"profiles `{int(top['profile_count'])}`, windows `{int(top['window_count'])}`. "
        f"Dominant failure type is `{top['dominant_failure_type']}` with mean excess "
        f"`{float(top['mean_excess_return']):.2f}pp`, worst excess "
        f"`{float(top['worst_excess_return']):.2f}pp`, max drawdown "
        f"`{float(top['max_max_dd']):.2f}%`, and mean position "
        f"`{float(top['mean_avg_pos']):.3f}`.",
        '',
        f'Stable feature-delta hypotheses: {top_feature_names}.',
        '',
        'Readout:',
        '',
        '- This is a shared OOS failure cluster, not an isolated artifact of the newest reward run.',
        '- If the top cluster stays underparticipation-heavy, the next training branch should not be another reward coefficient tweak. It should refresh participation behavior with supervised/BC targets or a fixed-actor re-entry overlay and then retest `2022`, `2023-2024`, and walk-forward.',
        '- Feature deltas remain hypotheses only; they need IC / single-signal / walk-forward gates before becoming promoted factors.',
    ]
    conclusion = '\n'.join(conclusion_lines)

display(Markdown(conclusion))
(CARD_DIR / 'runtime_conclusion.md').write_text(conclusion + '\n', encoding='utf-8')

## Runtime Conclusion

Decision: `next_training_target_reentry_participation_or_behavior_clone`.

Top shared bad-window cluster: `2025-04-04` .. `2025-07-22`, profiles `5`, windows `30`. Dominant failure type is `underparticipation` with mean excess `-160.50pp`, worst excess `-199.39pp`, max drawdown `16.32%`, and mean position `0.520`.

Stable feature-delta hypotheses: vol_of_vol_21, tracking_error_vol_21, decay_63, realized_vol_63, tracking_error_21, realized_vol_21.

Readout:

- This is a shared OOS failure cluster, not an isolated artifact of the newest reward run.
- If the top cluster stays underparticipation-heavy, the next training branch should not be another reward coefficient tweak. It should refresh participation behavior with supervised/BC targets or a fixed-actor re-entry overlay and then retest `2022`, `2023-2024`, and walk-forward.
- Feature deltas remain hypotheses only; they need IC / single-signal / walk-forward gates before becoming promoted factors.

984

## Runtime Conclusion

Decision: `next_training_target_reentry_participation_or_behavior_clone`.

Top shared bad-window cluster: `2025-04-04` .. `2025-07-22`, profiles `5`, windows `30`. Dominant failure type is `underparticipation` with mean excess `-160.50pp`, worst excess `-199.39pp`, max drawdown `16.32%`, and mean position `0.520`.

Stable feature-delta hypotheses: vol_of_vol_21, tracking_error_vol_21, decay_63, realized_vol_63, tracking_error_21, realized_vol_21.

Readout:

- This is a shared OOS failure cluster, not an isolated artifact of the newest reward run.
- If the top cluster stays underparticipation-heavy, the next training branch should not be another reward coefficient tweak. It should refresh participation behavior with supervised/BC targets or a fixed-actor re-entry overlay and then retest `2022`, `2023-2024`, and walk-forward.
- Feature deltas remain hypotheses only; they need IC / single-signal / walk-forward gates before becoming promoted factors.